# Evaluation Metrics for Generative Modeling on Spatial Transcriptomics
**Data**: paired spatial-transcriptomics slides (e.g. Mouse Brain Atlas / ABCA)

A runnable companion to the `nicheflow_eval` metric suite. Each section below explains one metric
and shows the **exact code to run it on its own**, against a real **target slide** and a set of
**generated cells**. The metrics evaluate how a generative model (e.g. the flow-matching models in
NicheFlow) reproduces gene expression **and** spatial coordinates simultaneously.

Run **§0 Environment setup** and **§0b Data loading** first; after that every metric section is
independent and can be run on its own. Inputs are plain arrays — no flow model, checkpoint, or
Hydra:

- `TargetSlide` — the real target slide: `x (N,P)` PCA expression, `pos (N,2)` coords, `ct (N,)`
  cell types, optional grid subsample.
- `GeneratedNiches` — `(B, N, D)` generated microenvironments, **centroid at point 0**.

## 0. Environment setup

Install the package (editable), then check which optional backends are present — some metrics need
extra deps: `torch`+`pot` for EMD/MMD, `squidpy` for Moran's I.

```bash
cd nicheflow-eval
uv venv && uv pip install -e .       # or:  pip install -e .
```

In [ ]:
import importlib
import numpy as np
import pandas as pd


def _has(mod):
    try:
        importlib.import_module(mod)
        return True
    except Exception:
        return False


print("optional backends present:")
print("  torch   (MMD / EMD / classifier):", _has("torch"))
print("  ot/pot  (EMD / Wasserstein)     :", _has("ot"))
print("  squidpy (Moran's I)             :", _has("squidpy"))

## 0a. Generate the cells to evaluate (bring your own model)

The metrics compare a real **target slide** against a set of **generated cells**. Generation is a
pluggable blackbox: run *any* model and write its output to a generated `.h5ad` that §0b loads.

Below we use the bundled **NicheFlow** adapter as a concrete example — it preprocesses the
source+target pair (shared PCA, radius-graph niches) and samples the flow. Needs the `[pipeline]`
extra (`pip install -e ../nicheflow_mba`) and a trained checkpoint. **If you already have a
generated `.h5ad`, skip to §0b.**

To use your own model, replace the marked block: generate cells and write them to `GENERATED_PATH`
in either layout —
- **niche-shaped** (`GeneratedNiches`): one row per cell with `obs['niche_id']` grouping each niche
  (centroid first); add `obsm['gt_x']`/`obsm['gt_pos']`/`obs['gt_ct']` to enable the niche metrics;
- **flat whole-slide** (`GeneratedSlide`): just `X` + `obsm['spatial']` (the niche metrics then skip).

In [ ]:
# NicheFlow as the example generator. Swap the marked block for your own model.
import os

SOURCE_H5AD    = "../data/source.h5ad"                  # source slide (raw genes + coords)
TARGET_H5AD    = "../data/target.h5ad"                  # target slide to generate
CHECKPOINT     = "../ckpts/NicheFlow_CFM_ABCA.ckpt"     # trained flow checkpoint
GENERATED_PATH = "../data/generated.h5ad"              # <- the file §0b loads

if _has("nicheflow") and os.path.exists(CHECKPOINT):
    # ---- your model goes here; this block is the NicheFlow example ----
    from nicheflow_eval.adapters.nicheflow import generate, preprocess_pair

    ds, _ = preprocess_pair(SOURCE_H5AD, TARGET_H5AD, n_pcs=50, cell_type_column="class")
    gen = generate(ds, CHECKPOINT, variant="cfm")      # samples the flow
    # -------------------------------------------------------------------
    gen.to_anndata().write_h5ad(GENERATED_PATH)        # niche-shaped generated cells
    print(f"wrote {GENERATED_PATH}: {gen.x.shape[0]} niches x {gen.x.shape[1]} points")
else:
    print(f"nicheflow/checkpoint unavailable -> §0b will load an existing {GENERATED_PATH}")

## 0b. Data loading

Inputs are **original AnnData (`.h5ad`) files** — raw gene expression + `obsm['spatial']`
coordinates. No preprocessed pickle is needed.

- **`target.h5ad`** — the target slide. `TargetSlide.from_anndata` reads `adata.X` (raw genes) and
  `obsm['spatial']`; pass `ct_key` for cell types and `n_pcs` to fit a shared PCA on the target.
- **`generated.h5ad`** — the generated cells, in one of two shapes:
  - **flat** (`GeneratedSlide`): `X` + `obsm['spatial']`, one row per cell — for whole-slide
    models. The label-free metrics run; the niche metrics are skipped.
  - **niche-shaped** (`GeneratedNiches`): flat rows with `obs['niche_id']` grouping each niche
    (centroid first); optional `gt_x`/`gt_pos`/`gt_ct` enable regression and the classifier
    metrics. (This notebook uses the niche shape.)

Target and generated must share one feature space — keep both as raw genes, or set `N_PCS` to fit a
PCA on the target and `.project(target.pca)` the generated cells into it. To go from a checkpoint
instead, see `nicheflow_eval.pipeline.run_pipeline` with a `generator` (the bundled
`nicheflow_eval.adapters.nicheflow.nicheflow_generator`, or your own model's).

In [ ]:
import os

import numpy as np

from nicheflow_eval import TargetSlide, GeneratedNiches

# Inputs are ORIGINAL AnnData files (raw genes + obsm['spatial']) — no preprocessed pickle.
TARGET_H5AD = "../data/target.h5ad"        # the target slide (raw genes + coords [+ obs cell types])
GENERATED_PATH = "../data/generated.h5ad"  # flat generated cells, or an .npz with x/pos[/gt_x/gt_pos/gt_ct]
CT_KEY = "class"                            # obs column with cell types (optional)
N_PCS = None                                # set e.g. 50 to fit a shared PCA on the target
SEED = 0

if not os.path.exists(TARGET_H5AD):
    raise FileNotFoundError(f"Target slide not found: {TARGET_H5AD}. Point TARGET_H5AD at a .h5ad slide.")
if not os.path.exists(GENERATED_PATH):
    raise FileNotFoundError(f"Generated cells not found: {GENERATED_PATH}.")

# --- target slide (raw AnnData) ---
target = TargetSlide.from_anndata(TARGET_H5AD, ct_key=CT_KEY, n_pcs=N_PCS)
print(f"target: {target.x.shape[0]} cells, {target.x.shape[1]} features, {target.n_classes} cell types")

# --- generated cells ---
if GENERATED_PATH.endswith(".h5ad"):
    generated = GeneratedNiches.from_anndata(GENERATED_PATH)
else:
    npz = np.load(GENERATED_PATH)
    extra = {k: npz[k] for k in ("gt_x", "gt_pos", "gt_ct") if k in npz}
    generated = GeneratedNiches(x=npz["x"], pos=npz["pos"], **extra)
generated = generated.project(target.pca)   # no-op unless N_PCS set (shared-basis projection)

print(f"generated: {generated.x.shape[0]} niches x {generated.x.shape[1]} points")


## 1. Pointwise regression (MSE / MAE) — original NicheFlow

The most basic metric: per matched cell, the error between the final generated state and its
ground-truth target cell, scored separately for gene expression (`x`) and coordinates (`pos`).
Lower = better. Computed by `regression_metrics` in `nicheflow_eval.metrics.distances`.

- `test/x/mse` — **mean squared error** of predicted vs. true **gene expression** (lower = better).
- `test/x/mae` — **mean absolute error** of predicted vs. true **gene expression** (lower = better).
- `test/pos/mse` — MSE of predicted vs. true **spatial coordinates** (lower = better).
- `test/pos/mae` — MAE of predicted vs. true **coordinates** (lower = better).

These reward *exact per-cell* prediction (complementary to the distribution-level EMD/MMD below).
Needs the matched ground-truth microenvironment per generated niche (`generated.gt_x/gt_pos`).
Note NicheFlow reports MSE/MAE but **not** R².

In [ ]:
from nicheflow_eval.metrics.distances import regression_metrics

if generated.gt_x is not None and generated.gt_pos is not None:
    res = regression_metrics(generated.x, generated.gt_x, generated.pos, generated.gt_pos, prefix="test")
    for k, v in res.items():
        print(f"{k:14s} {v:.4f}")
else:
    print("Regression needs matched ground truth: set generated.gt_x / generated.gt_pos "
          "(the true target microenvironment per generated niche).")

## 2. EMD

Earth-mover (Wasserstein) distance between the **pooled generated cells** and **all real target
cells**, computed **separately** for gene expression (`x`, 50-d PCA) and spatial coordinates
(`pos`, 2-d) — they live on different scales, so they are scored independently. Lower = the
generated distribution is closer to the real one. Computed with exact OT (POT `emd2`) in
`nicheflow_eval.metrics.distribution.ot_distance`.

- `test/ot_w1/x` — **Wasserstein-1** distance on **gene expression** (lower = better).
- `test/ot_w1/pos` — Wasserstein-1 on **coordinates** (lower = better).
- `test/ot_w2/x` — **Wasserstein-2** on **expression** (penalizes large transport more than W1; lower = better). *Checkpoint-selection metric (`val/ot_w2/x`).*
- `test/ot_w2/pos` — Wasserstein-2 on **coordinates** (lower = better).

Caveat: scoring `x` and `pos` separately cannot detect a wrong expression↔position **coupling** —
that is what the C2ST (§8) adds.

In [ ]:
from nicheflow_eval.metrics.distribution import ot_distance

# pooled generated cloud vs all real target cells, scored separately for x and pos
real_x, real_pos = target.x, target.pos
gen_x, gen_pos = generated.flat_x, generated.flat_pos

print("test/ot_w1/x  ", round(ot_distance(real_x,   gen_x,   power=1, seed=SEED), 4))
print("test/ot_w1/pos", round(ot_distance(real_pos, gen_pos, power=1, seed=SEED), 4))
print("test/ot_w2/x  ", round(ot_distance(real_x,   gen_x,   power=2, seed=SEED), 4))
print("test/ot_w2/pos", round(ot_distance(real_pos, gen_pos, power=2, seed=SEED), 4))

## 3. MMD

Maximum Mean Discrepancy (squared, RBF kernel) between the pooled generated cells and all real
target cells — a kernel two-sample distance — again scored **separately** for expression and
coordinates. Lower = distributions more similar. Computed with
`nicheflow_eval.metrics.distribution.mmd2_rbf`.

- `test/mmd2/x` — squared MMD on **gene expression** (lower = better).
- `test/mmd2/pos` — squared MMD on **coordinates** (lower = better).

Like EMD, it compares the `x` and `pos` **marginals** independently. (The unbiased U-statistic can
dip slightly negative when the distributions match — that is expected, do not clip.)

In [ ]:
from nicheflow_eval.metrics.distribution import mmd2_rbf

print("test/mmd2/x  ", round(mmd2_rbf(target.x,   generated.flat_x,   seed=SEED), 5))
print("test/mmd2/pos", round(mmd2_rbf(target.pos, generated.flat_pos, seed=SEED), 5))

## 4. Point-to-Shape distance

For each generated cell, distance to its nearest real target cell.

- `test/psd/mean` — average nearest-real-neighbour distance of generated cells, i.e. how close generations land to the real manifold (lower = better).
- `test/psd/max` — worst-case (largest) nearest-neighbour distance, capturing the most off-manifold generated cell (lower = better).

In [ ]:
from nicheflow_eval.metrics.distances import point_to_shape

res = point_to_shape(generated.flat_pos, target.pos, prefix="test")
for k, v in res.items():
    print(f"{k:14s} {v:.4f}")

## 5. Shape-to-Point distance

The reverse direction: for each real target cell, distance to its nearest generated cell (a coverage measure).

- `test/spd/mean` — average distance from real cells to the nearest generated cell, i.e. how well generations **cover** the real distribution (lower = better).
- `test/spd/max` — largest such distance, flagging the most poorly covered real region (lower = better).

In [ ]:
from nicheflow_eval.metrics.distances import shape_to_point

res = shape_to_point(generated.flat_pos, target.pos, prefix="test")
for k, v in res.items():
    print(f"{k:14s} {v:.4f}")

## 6. Cell-type Classifier-based metric (neutral held-out classifier)

A **neutral** SetTransformer spatial classifier — trained on a **held-out same-mouse slide**
(`abca_aligned_clf.pkl`, *neither* source nor target, projected into the source+target PCA basis +
label space) — labels **both** the generated niche and its **paired real target niche** (`gt_*`,
same centroid). Because the same annotator scores both sides and it never saw the target slide,
this avoids the leakage/circularity of the old nearest-real-cell pseudo-label.
`cell_type_concordance` (`nicheflow_eval.metrics.concordance`) reports:

- `test/ct/f1` — weighted multiclass **F1** of the classifier's labels on generated vs. real niches (higher = better).
- `test/ct/acc` — **accuracy** of the same agreement (higher = better).
- `test/ct/prop_kl` — **KL divergence** between the generated/real **label-proportion** histograms (lower = better).
- `test/ct/prop_tv` — **total-variation** distance between those proportions (lower = better).
- `test/ct/prop_jsd` — **Jensen-Shannon** divergence between those proportions (symmetric; lower = better).

**f1 / acc** = per-niche agreement (is each generated niche labelled like the real one?);
**prop_kl / prop_tv / prop_jsd** = population-level cell-type composition fidelity.

> Needs (a) the paired real niches `generated.gt_x / gt_pos` and (b) a classifier trained on the
> held-out slide:
> `python -m nicheflow_eval.classifier.train experiment=classifier/abca_spatial`
> (its `data.datamodule.data_fp` points at `abca_aligned_clf.pkl`). Point `CLASSIFIER_CKPT` at the
> resulting checkpoint. Loading via `load_spatial_classifier` also carries the training
> `n_neighbors` onto the net, so eval builds identically sized niches.

In [ ]:
import torch

from nicheflow_eval.classifier.nets import SpatialCTClassifierNet
from nicheflow_eval.metrics._common import load_spatial_classifier
from nicheflow_eval.metrics.concordance import cell_type_concordance

CLASSIFIER_CKPT = "../ckpts/Classifier_Spatial_ABCA_SetTransformer.ckpt"

clf = SpatialCTClassifierNet(
    input_dim=target.x.shape[1],
    output_dim=target.n_classes,
    hidden_dim=64,
    coord_dim=2,
    num_heads=4,
    mask_centroid=True,
)

if os.path.exists(CLASSIFIER_CKPT):
    load_spatial_classifier(clf, torch.load(CLASSIFIER_CKPT, map_location="cpu"))  # +n_neighbors
    print("loaded trained neutral classifier")
else:
    print("no checkpoint -> UNTRAINED classifier (numbers are meaningless; "
          "train one on abca_aligned_clf.pkl first).")

if generated.gt_x is not None and generated.gt_pos is not None:
    res = cell_type_concordance(
        generated.x, generated.pos, generated.gt_x, generated.gt_pos, clf,
        prefix="test", spatial=True, n_classes=target.n_classes,   # n_neighbors comes from the ckpt
    )
    for k, v in res.items():
        print(f"{k:18s} {v:.4f}")
else:
    print("concordance needs the paired real niches generated.gt_x / generated.gt_pos.")

## 6b. Classifier accuracy gap

A complementary read on the same classifier: run it on the **real** target niches and on the **generated** niches, each scored against the **true** centroid labels (`gt_ct`). If generation is realistic the classifier should be about as accurate on both, so a small `ct/acc_gap = |acc_real - acc_gen|` is good.


In [ ]:
from nicheflow_eval.metrics.classifier_gap import classifier_accuracy_gap

if generated.gt_x is not None and generated.gt_ct is not None:
    res = classifier_accuracy_gap(
        generated.x, generated.pos, generated.gt_x, generated.gt_pos, generated.gt_ct, clf,
        prefix="test", spatial=True,
    )
    for k, v in res.items():
        print(f"{k:18s} {v:.4f}")
else:
    print("accuracy gap needs generated.gt_x / gt_pos and true centroid labels generated.gt_ct.")


## 7. Moran's I Score

Moran's I measures whether a feature is spatially **clustered** (nearby cells have similar values).
In the PC space (e.g. `n_pcs = 50`), `morans_compare` (`nicheflow_eval.metrics.morans`) computes
it **per PC** over **all** generated cells (every cell of every niche) and over the full real
target slide (same squidpy kNN graph, `n_neighs=6`), then compares the two per-PC vectors,
**variance-weighted** by real per-PC variance.

| Metric | What it compares | Reference (ABCA) |
|---|---|---|
| `moran/corr` ↑ | weighted Pearson corr of per-PC `I_real` vs `I_gen` across PCs | 0.952 (0.948–0.954) |
| `moran/mae` ↓ | weighted mean `\|I_real − I_gen\|` over PCs | 0.039 (0.037–0.041) |
| `moran/real_mean` | weighted mean Moran's I of the **real** slide (diagnostic) | 0.0665 |
| `moran/gen_mean` | weighted mean Moran's I of the **generated** slide (diagnostic) | 0.081 |

### What each Moran metric means
- **`moran/corr`** — across the PCs, do the PCs that are spatially clustered in the real slide come
  out clustered in the generated slide too? 1.0 = the **relative** clustering pattern is perfectly
  reproduced; 0 = unrelated.
- **`moran/mae`** — the average absolute gap between real and generated Moran's I per PC (after
  variance weighting). 0 = identical clustering strength PC-by-PC.
- **`moran/real_mean` / `moran/gen_mean`** — diagnostics, not a score: the overall clustering level
  of each slide. Comparing them tells you the *direction* of any mismatch (over- vs under-clustered).

In [ ]:
from nicheflow_eval.metrics.morans import morans_compare

# Moran's I over ALL generated cells (every cell of every niche) vs the full real target slide.
res = morans_compare(
    generated.flat_x, generated.flat_pos, target.x, target.pos,
    prefix="test", n_neighs=6, seed=SEED,
)
for k, v in res.items():
    print(f"{k:18s} {v:.4f}")


## 8. Classifier Two-Sample Test

C2ST trains a classifier to tell **real** target cells from **generated** ones; **no cell-type
labels or pseudo-labels** are used, so it sidesteps the caveat of the concordance metrics (§6).
Each score is a held-out accuracy/AUC: **~0.5 = indistinguishable (good), ~1.0 = trivially
separable (bad)**. Computed by `c2st_metrics` (`nicheflow_eval.metrics.c2st`).

| Metric | What it compares | Reference (ABCA) |
|---|---|---|
| `c2st/acc` | per-cell **joint `[expression \| position]`** | 0.598 (0.590–0.608) |
| `c2st/auc` | "" (ROC-AUC) | 0.633 (0.626–0.636) |
| `c2st/pos_acc` | per-cell **position only** (diagnostic) | 0.561 (0.549–0.578) |

### What each C2ST metric means
- **`c2st/acc`** — accuracy of an MLP distinguishing real vs generated using the **joint**
  `[expression | position]` vector: the **co-generation** test (detects a wrong expression↔position
  coupling the separate MMD/EMD scores cannot). 0.5 = indistinguishable.
- **`c2st/auc`** — the same test by ROC-AUC (threshold-independent; robust to class imbalance).
- **`c2st/pos_acc`** — diagnostic C2ST on **coordinates only**; surfaces spatial problems that the
  ~50-d joint vector might dilute. 0.5 = the generated spatial *marginal* matches real.

### Significance test (label permutation)
Set `n_perm > 0` (e.g. 100) to test whether the joint AUC is **significantly above chance**: pool
real + generated, shuffle the labels `n_perm`×, recompute the AUC to build a null. Adds
`c2st/sig_auc`, `c2st/sig_null_p95`, `c2st/sig_pval` — **significant if `sig_auc > sig_null_p95`
(pval < 0.05)**.

In [ ]:
from nicheflow_eval.metrics.c2st import c2st_metrics

res = c2st_metrics(
    target.x, target.pos,
    generated.flat_x, generated.flat_pos,
    prefix="test",
    n_folds=5,
    n_perm=0,                            # set to 100 to add the label-permutation significance test
    seed=SEED,
)
for k, v in res.items():
    print(f"{k:20s} {v:.4f}")

## 9. Run the whole suite at once

`evaluate` runs every applicable group and returns a flat `{test/group/metric: value}` dict (the
columns the NicheFlow result CSVs use). Groups whose inputs are missing are listed under
`_skipped`. Add `"regression"` once `generated.gt_*` is set and `"concordance"` once you pass a
trained `classifier=`.

In [ ]:
from nicheflow_eval import evaluate

groups = ("psd", "spd", "distribution", "c2st", "moran")
all_res = evaluate(target, generated, groups=groups, seed=SEED)
print("skipped:", all_res.pop("_skipped"))
pd.DataFrame(sorted(all_res.items()), columns=["metric", "value"])